In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
import kagglehub

# AIJack Imports
from aijack.collaborative.fedavg import FedAVGClient, FedAVGServer, FedAVGAPI
from aijack.attack.inversion import GradientInversion_Attack
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from aijack.collaborative.fedavg import FedAVGClient, FedAVGServer, FedAVGAPI


In [ ]:

path = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database")

print("Path to dataset files:", path)

Path to dataset files: /Users/flaviafuscaldi/.cache/kagglehub/datasets/tawsifurrahman/covid19-radiography-database/versions/5


In [7]:
#Hyperparameters
training_batch_size = 64
test_batch_size = 64
num_rounds = 5
lr = 0.001
seed = 0
client_size = 2
criterion = F.nll_loss


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def fix_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

fix_seed(seed)

# Path from your kagglehub download
data_dir = os.path.join(path, 'COVID-19_Radiography_Dataset')

In [8]:
# Transforms: Medical standard 224x224
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# Partition indices for 2 Clients (Hospital A & B)
labels = np.array(full_dataset.targets)
h0_idx = np.where((labels == 0) | (labels == 3))[0] # COVID & Pneumonia
h1_idx = np.where((labels == 1) | (labels == 2))[0] # Normal & Opacity

train_loaders = [
    DataLoader(Subset(full_dataset, h0_idx), batch_size=4, shuffle=True),
    DataLoader(Subset(full_dataset, h1_idx), batch_size=4, shuffle=True)
]

In [9]:
def get_medical_model():
    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, 4)
    return model.to(device)

global_model = get_medical_model()
criterion = nn.CrossEntropyLoss()

/Users/flaviafuscaldi/Desktop/fl_thesis/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/flaviafuscaldi/Desktop/fl_thesis/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [10]:
# Initialize 2 Clients
clients = [FedAVGClient(get_medical_model(), user_id=i) for i in range(2)]
server = FedAVGServer(clients, global_model)

# The "Spy" Attack Configuration (Modify these for your experiments)
config = {
    "num_iteration": 500,
    "lr": 1.0,
    "optimizer_class": torch.optim.LBFGS,
    "distancename": "l2",
    "tv_reg_coef": 0.01  # Important for X-ray smoothness
}

# Setup the Attack
medical_spy = GradientInversion_Attack(
    global_model, 
    x_shape=(3, 224, 224), 
    **config
)

In [12]:
# 1. Simulate a local training step at Hospital A
client_id = 0
inputs, targets = next(iter(train_loaders[client_id]))
inputs, targets = inputs.to(device), targets.to(device)

# 2. Get the "Victim" Gradients (CORRECTED)
# Instead of server.model, we use global_model (the base for the server)
clients[client_id].model.load_state_dict(global_model.state_dict())
optimizer = optim.SGD(clients[client_id].model.parameters(), lr=0.01)

optimizer.zero_grad()
# Forward pass
output = clients[client_id].model(inputs)
loss = criterion(output, targets)
# Backward pass
loss.backward()

# Extract gradients for the attack
target_gradients = [p.grad.detach() for p in clients[client_id].model.parameters() if p.grad is not None]

# 3. Server-side Attack
print(f"Server attempting to invert gradients from Hospital {client_id}...")
# Note: AIJack's group_attack returns the reconstruction results
reconstruction = medical_spy.group_attack(target_gradients, batch_size=1)

Server attempting to invert gradients from Hospital 0...
worker_id=0: iter=10: 666.2227783203125, (best_iter=10: 666.2227783203125)
worker_id=1: iter=10: 667.2078857421875, (best_iter=10: 667.2078857421875)
worker_id=2: iter=10: 668.0654907226562, (best_iter=8: 667.232177734375)
worker_id=3: iter=10: 807.3189086914062, (best_iter=10: 807.3189086914062)
worker_id=4: iter=10: 666.5352783203125, (best_iter=10: 666.5352783203125)
worker_id=0: iter=20: 668.814208984375, (best_iter=11: 665.9076538085938)
worker_id=1: iter=20: 665.7845458984375, (best_iter=19: 665.680419921875)
worker_id=2: iter=20: 668.3615112304688, (best_iter=15: 666.9043579101562)
worker_id=3: iter=20: 820.9683227539062, (best_iter=10: 807.3189086914062)
worker_id=4: iter=20: 666.3756713867188, (best_iter=18: 666.296630859375)
worker_id=0: iter=30: 666.3912963867188, (best_iter=11: 665.9076538085938)
worker_id=1: iter=30: 667.6478271484375, (best_iter=19: 665.680419921875)
worker_id=2: iter=30: 668.0834350585938, (best_it

KeyboardInterrupt: 